# Notebook 09-B — GridSearchCV Mejorado con Malla Ampliada (Literatura)

## Objetivo
Repetir el benchmark de modelos lineales del Notebook 09 con una **malla de
hiperparametros ampliada** segun la literatura cientifica, para explorar si
combinaciones mas agresivas de `max_features` y regularizacion (`C`) mejoran
el F1-Score Macro.

### Diferencias respecto al Notebook 09 original

| Parametro | NB-09 (original) | NB-09-B (este) |
|---|---|---|
| `max_features` | [5000, 10000] | [5000, 10000, **20000**] |
| `C` (SVM) | [0.1, 1.0, 10.0] | [0.1, 1.0, 10.0, **100.0**] |
| `C` (Logistica) | [0.1, 1.0, 10.0] | [0.1, 1.0, 10.0, **100.0**] |
| Combinaciones SVM | 6 | **12** |
| Combinaciones Log. | 6 | **12** |
| Total fits (x5 folds) | 60 | **120** |

> `max_features=20000` se incorpora siguiendo el limite usado en estudios
> comparativos chilenos de NLP. `C=100.0` explora regularizacion mas debil
> (modelo mas complejo) para buscar el punto optimo.

### Condiciones identicas al NB-09
- **Pipeline**: `RandomUnderSampler` -> `TF-IDF` -> Clasificador
- **CV**: `StratifiedKFold(K=5, shuffle=True, random_state=42)`
- **Metrica**: `f1_macro`
- **Monitoreo**: `memory_profiler` + cronometro
- **Columna**: `texto_lineal`

### Formato de salida esperado
```
SVM:            F1-Score: 0.XX | Tiempo: XX min | RAM: X.XX GB
Reg. Logistica: F1-Score: 0.XX | Tiempo: XX min | RAM: X.XX GB
```

## Celda 0 — Verificacion de memory_profiler

In [ ]:
import memory_profiler
print(f"memory_profiler version: {memory_profiler.__version__}")

## Celda 1 — Importaciones

In [ ]:
import pandas as pd
import time
import warnings
import gc

# Monitoreo de RAM
from memory_profiler import memory_usage

# scikit-learn
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.base import clone

# imbalanced-learn
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings('ignore')
print("Librerias cargadas correctamente.")

## Celda 2 — Carga del Corpus Completo

In [ ]:
INPUT_PATH = 'data/train_val_set.csv'

df = pd.read_csv(INPUT_PATH, sep=';')
df['texto_lineal'] = df['texto_lineal'].fillna('')

X = df[['texto_lineal']]
y = df['label']

print(f"Corpus cargado.")
print(f"   Total de registros : {len(df):,}")
print(f"   Distribucion de clases:\n{y.value_counts().to_string()}")

## Celda 3 — Pipeline Base y Validacion Cruzada Estratificada

In [ ]:
# Validacion Cruzada Estratificada (K=5) — misma semilla que NB-09
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Submuestreo para balance de clases (identico al NB-09)
undersampler = RandomUnderSampler(random_state=42)

# Vectorizador TF-IDF sobre la columna 'texto_lineal'
preprocessor = ColumnTransformer(
    transformers=[('tfidf', TfidfVectorizer(), 'texto_lineal')]
)

# Pipeline molde (el clasificador se reemplaza por cada malla)
pipeline_base = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LogisticRegression())  # placeholder
])

print("Pipeline base y CV estratificado (K=5) configurados.")

## Celda 4 — Helper: funcion de entrenamiento + medicion de RAM

Se usa `memory_profiler.memory_usage()` con `interval=0.1` s para registrar el
**pico maximo de RAM (en MB)** durante toda la ejecucion del GridSearchCV.
Despues se convierte a GB para el reporte final.

Ademas se registra el ranking completo de todas las combinaciones para
poder analizar el impacto de cada hiperparametro.

In [ ]:
def entrenar_con_metricas(grid_search_obj, X_data, y_data, nombre_modelo):
    """
    Ejecuta grid_search_obj.fit(X_data, y_data) midiendo:
      - Pico de RAM (memory_profiler) en GB
      - Tiempo total en minutos
    Retorna: (best_score, tiempo_min, ram_gb)
    """
    print(f"\n{'='*60}")
    print(f"  Entrenando: {nombre_modelo}")
    print(f"  (GridSearchCV K=5 — malla ampliada)")
    n_combos = 1
    for p in grid_search_obj.param_grid:
        combo = 1
        for v in p.values():
            combo *= len(v) if isinstance(v, list) else 1
        n_combos += combo
    print(f"  Combinaciones: {n_combos - 1} x 5 folds = {(n_combos - 1) * 5} fits")
    print(f"{'='*60}")

    def _fit():
        grid_search_obj.fit(X_data, y_data)

    t_inicio = time.time()

    muestras_ram = memory_usage(
        (_fit, [], {}),
        interval=0.1,
        include_children=True,
        max_usage=False,
        retval=False
    )

    t_fin = time.time()

    pico_ram_mb = max(muestras_ram)
    pico_ram_gb = pico_ram_mb / 1024
    tiempo_min  = (t_fin - t_inicio) / 60
    best_score  = grid_search_obj.best_score_

    print(f"\n  Entrenamiento finalizado.")
    print(f"  Mejor configuracion : {grid_search_obj.best_params_}")
    print(f"  Mejor F1-Score Macro: {best_score:.4f}")
    return best_score, tiempo_min, pico_ram_gb

## Celda 5 — GridSearchCV para SVM (LinearSVC) — Malla Ampliada

Respecto al NB-09:
- Se agrega `max_features=20000` (vocabulario mas rico)
- Se agrega `C=100.0` (regularizacion mas debil, modelo mas complejo)

In [ ]:
# Malla de hiperparametros ampliada para SVM (segun literatura)
param_grid_svm = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000, 20000],
        'classifier': [LinearSVC(max_iter=2000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0, 100.0]
    }
]

grid_svm = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_svm,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_svm, tiempo_svm, ram_svm = entrenar_con_metricas(
    grid_svm, X, y, "SVM — LinearSVC (malla ampliada)"
)

gc.collect()
print("\nMemoria liberada. Listo para Regresion Logistica.")

## Celda 6 — GridSearchCV para Regresion Logistica — Malla Ampliada

In [ ]:
# Malla de hiperparametros ampliada para Regresion Logistica
param_grid_log = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000, 20000],
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0, 100.0]
    }
]

grid_log = GridSearchCV(
    estimator=clone(pipeline_base),
    param_grid=param_grid_log,
    cv=cv_estratificado,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

f1_log, tiempo_log, ram_log = entrenar_con_metricas(
    grid_log, X, y, "Regresion Logistica (malla ampliada)"
)

gc.collect()
print("\nMemoria liberada.")

## Celda 7 — Ranking Completo de Combinaciones

Para analizar el impacto de cada hiperparametro se muestran las
12 combinaciones ordenadas por F1-Score (descendente).

In [ ]:
import numpy as np

def mostrar_ranking(grid_obj, nombre):
    """
    Extrae cv_results_ del GridSearchCV y muestra un ranking
    de todas las combinaciones ordenado por F1 descendente.
    """
    resultados_cv = pd.DataFrame(grid_obj.cv_results_)
    cols = [c for c in resultados_cv.columns if c.startswith('param_') or c in ['mean_test_score', 'std_test_score', 'rank_test_score']]
    ranking = resultados_cv[cols].sort_values('rank_test_score')

    # Renombrar columnas para legibilidad
    rename_map = {}
    for c in ranking.columns:
        if 'max_features' in c:
            rename_map[c] = 'max_features'
        elif 'classifier__C' in c:
            rename_map[c] = 'C'
        elif c == 'mean_test_score':
            rename_map[c] = 'F1 Macro (mean)'
        elif c == 'std_test_score':
            rename_map[c] = 'F1 Macro (std)'
        elif c == 'rank_test_score':
            rename_map[c] = 'Rank'

    ranking = ranking.rename(columns=rename_map)
    # Eliminar la columna del clasificador (es la misma para todas las filas)
    ranking = ranking.drop(columns=[c for c in ranking.columns if 'classifier' in c.lower() and c not in rename_map.values()], errors='ignore')

    print(f"\n{'='*60}")
    print(f"  Ranking completo: {nombre}")
    print(f"{'='*60}")
    display(ranking.reset_index(drop=True))

mostrar_ranking(grid_svm, 'SVM — LinearSVC')
mostrar_ranking(grid_log, 'Regresion Logistica')

## Celda 8 — Tabla Comparativa: NB-09 (original) vs NB-09-B (ampliado)

In [ ]:
print("\n" + "=" * 75)
print("  RESULTADOS — GridSearchCV K=5 (Malla Ampliada segun Literatura)")
print("=" * 75)
print(f"  SVM:             F1-Score: {f1_svm:.2f} | Tiempo: {tiempo_svm:.1f} min | RAM: {ram_svm:.2f} GB")
print(f"  Reg. Logistica:  F1-Score: {f1_log:.2f} | Tiempo: {tiempo_log:.1f} min | RAM: {ram_log:.2f} GB")
print("=" * 75)

# Tabla comparativa con resultados del NB-09 original (hardcoded del output)
resultados = pd.DataFrame({
    'Modelo':         ['SVM (LinearSVC)', 'SVM (LinearSVC)',
                       'Reg. Logistica', 'Reg. Logistica'],
    'Notebook':       ['09 (original)', '09-B (ampliado)',
                       '09 (original)', '09-B (ampliado)'],
    'F1-Score Macro':  [0.9035, round(f1_svm, 4),
                        0.9041, round(f1_log, 4)],
    'Tiempo (min)':    [0.48, round(tiempo_svm, 2),
                        0.44, round(tiempo_log, 2)],
    'RAM Pico (GB)':   [2.052, round(ram_svm, 3),
                        2.100, round(ram_log, 3)],
    'Mejor C':         [1.0, grid_svm.best_params_.get('classifier__C', 'N/A'),
                        10.0, grid_log.best_params_.get('classifier__C', 'N/A')],
    'max_features':    [5000, grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A'),
                        5000, grid_log.best_params_.get('vectorizer__tfidf__max_features', 'N/A')],
    'Combinaciones':   [6, 12, 6, 12]
})

print("\n  Tabla comparativa NB-09 vs NB-09-B:")
display(resultados.set_index(['Modelo', 'Notebook']))

## Celda 9 — Exportar Resultados

In [ ]:
import joblib

# --- Solo los resultados de la malla ampliada ---
resumen_ampliado = pd.DataFrame({
    'Modelo':          ['SVM (LinearSVC)', 'Reg. Logistica'],
    'F1-Score Macro':  [round(f1_svm, 4), round(f1_log, 4)],
    'Tiempo (min)':    [round(tiempo_svm, 2), round(tiempo_log, 2)],
    'RAM Pico (GB)':   [round(ram_svm, 3), round(ram_log, 3)],
    'Mejor C':         [
        grid_svm.best_params_.get('classifier__C', 'N/A'),
        grid_log.best_params_.get('classifier__C', 'N/A')
    ],
    'max_features':    [
        grid_svm.best_params_.get('vectorizer__tfidf__max_features', 'N/A'),
        grid_log.best_params_.get('vectorizer__tfidf__max_features', 'N/A')
    ],
    'Malla':           ['ampliada', 'ampliada']
})

# CSV con resultados
resumen_ampliado.to_csv('data/resultados_lineales_malla_ampliada.csv', index=False, sep=';')
print("Resultados guardados en: data/resultados_lineales_malla_ampliada.csv")

# Rankings completos
cv_svm_df = pd.DataFrame(grid_svm.cv_results_)
cv_log_df = pd.DataFrame(grid_log.cv_results_)
cv_svm_df.to_csv('data/cv_results_svm_ampliada.csv', index=False, sep=';')
cv_log_df.to_csv('data/cv_results_log_ampliada.csv', index=False, sep=';')
print("Rankings CV completos guardados.")

# Modelos entrenados (refit=True -> ya estan ajustados al corpus completo)
joblib.dump(grid_svm.best_estimator_, 'data/best_svm_ampliada.joblib')
joblib.dump(grid_log.best_estimator_, 'data/best_log_ampliada.joblib')
print("Modelos best_estimator_ exportados como .joblib.")

print("\nExportacion completa.")